# Lecture 3 — Sampling, Quantization & Interpolation
**Week 3 | Spring 2026**  
**Instructor:** Dr. Hugo Guillen Ramirez  
**Course:** Introduction to Image Analysis

---

### How to use this notebook
This notebook is a **follow-along companion** to the lecture slides.  
Each section mirrors the slide deck. Run cells sequentially; every demo is self-contained.

**Exercises** are marked with ✏️ and should take **10–15 min** each.

---
## Setup & Imports
Run this cell first — everything else depends on it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter, laplace
from scipy.interpolate import RectBivariateSpline
from scipy.ndimage import zoom as scipy_zoom
from scipy.signal import fftconvolve
import time, warnings
warnings.filterwarnings('ignore')

try:
    from skimage import data as skdata
    from skimage.transform import resize as sk_resize
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print('scikit-image not found — using synthetic test images.')

plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'font.family': 'DejaVu Sans'})

# ── Load a standard greyscale test image ──────────────────────────────────────
def load_test_image(size=(256, 256)):
    """Return a greyscale float32 image in [0, 1]."""
    if HAS_SKIMAGE:
        img = skdata.camera()
        img = sk_resize(img, size, anti_aliasing=True)
    else:
        y, x = np.ogrid[:size[0], :size[1]]
        cy, cx = size[0] / 2, size[1] / 2
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        img = 0.5 + 0.5 * np.sin(r * 0.5)
    return img.astype(np.float32)

IMG = load_test_image()
print(f'Test image ready: shape={IMG.shape}, dtype={IMG.dtype}, '
      f'range=[{IMG.min():.2f}, {IMG.max():.2f}]')
plt.imshow(IMG, cmap='gray'); plt.title('Test image'); plt.axis('off'); plt.show()

---
# Section 1 — Sampling

## 1.1 Fourier Transform Refresher

The **1-D Fourier Transform** decomposes a signal into its frequency components:

$$F(\omega) = \int_{-\infty}^{\infty} f(x)\, e^{-2\pi i \omega x}\, dx$$

**Key property for sampling:** multiplication in the spatial domain $\Leftrightarrow$ convolution in the frequency domain.

### Rect → Sinc Fourier pair

A rectangular pulse of width $W$ transforms to a sinc:
$$\mathrm{rect}_W(x) \;\xrightarrow{\mathcal{F}}\; W\,\mathrm{sinc}(W\omega)$$

Wider aperture $\Leftrightarrow$ narrower spectrum.

In [ ]:
# ── Rect → Sinc Fourier pair ─────────────────────────────────────────────────
omega = np.linspace(-6, 6, 2000)

fig, ax = plt.subplots(figsize=(9, 4))
for W in [0.5, 1.0, 2.0]:
    F = W * np.sinc(W * omega)          # np.sinc(x) = sin(πx)/(πx)
    ax.plot(omega, F, label=f'W = {W}')

ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Frequency ω'); ax.set_ylabel('F(ω)')
ax.set_title('Fourier Transform of rect_W(x) → W · sinc(Wω)\n'
             'Wider aperture = narrower spectrum')
ax.legend(); plt.tight_layout(); plt.show()

## 1.2 Sampling in the Frequency Domain

Sampling a continuous signal $f(x)$ with spacing $\Delta x$ replicates its spectrum at intervals of $f_s = 1/\Delta x$.

If the copies **overlap**, we get **aliasing** — false low-frequency content injected into the signal.

In [ ]:
# ── Spectral replication: aliasing visualisation ─────────────────────────────
omega = np.linspace(-4, 4, 4000)
omega_max = 0.8

def original_spectrum(w, wmax):
    return np.maximum(0, 1 - np.abs(w) / wmax)

F_orig = original_spectrum(omega, omega_max)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, fs, title in zip(axes, [3.0, 1.2],
        ['Adequate  (fs=3.0 > 2×0.8)  — NO aliasing',
         'Inadequate (fs=1.2 < 2×0.8) — ALIASING']):
    ax.fill_between(omega, original_spectrum(omega, omega_max),
                    alpha=0.4, color='steelblue', label='Original F(ω)')
    for k in [-1, 1]:
        shifted = original_spectrum(omega - k * fs, omega_max)
        ax.fill_between(omega, shifted, alpha=0.35, color='tomato',
                        label='Copy' if k == 1 else None)
    ax.axvline(fs / 2,  ls='--', color='k', lw=0.8, label='Nyquist freq')
    ax.axvline(-fs / 2, ls='--', color='k', lw=0.8)
    ax.set_title(title); ax.set_xlabel('ω'); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 1.3 Nyquist–Shannon Sampling Theorem

> A band-limited signal with highest frequency $f_{\max}$ can be **perfectly reconstructed** from its samples if and only if
> $$f_s \geq 2\,f_{\max} \quad\Longleftrightarrow\quad \Delta x \leq \frac{1}{2\,f_{\max}}$$

Below: an 8 Hz signal sampled at three different rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1-D Nyquist demo ─────────────────────────────────────────────────────────
t  = np.linspace(0, 1, 2000)
fs = 8                                # signal frequency

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
configs = [
    (40,  'steelblue', 'Adequate (fs=40 > 16)'),
    (16,  'seagreen',  'Borderline (fs=16 = 2×8)'),
    (9,   'tomato',    'Under-sampled (fs=9 < 16) → ALIAS'),
]

for ax, (fs_s, col, title) in zip(axes, configs):
    t_s = np.arange(0, 1, 1 / fs_s)
    s_s = np.sin(2 * np.pi * fs * t_s)
    ax.plot(t, np.sin(2 * np.pi * fs * t), 'steelblue', lw=1, alpha=0.5)
    ax.stem(t_s, s_s, linefmt=col, markerfmt='o', basefmt=' ')
    
    if fs_s == 9:
        # Generalized alias formula that preserves the sign/phase
        f_alias = fs - fs_s * round(fs / fs_s)
        
        # Now we just plug f_alias straight into the sine function
        ax.plot(t, np.sin(2 * np.pi * f_alias * t), 'tomato', ls='--',
                lw=1.5, label=f'Alias at {f_alias} Hz')
        ax.legend(fontsize=8)
        
    ax.set_title(title); ax.set_xlabel('t')

plt.suptitle('Nyquist–Shannon: 8 Hz signal at different sampling rates',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 1.4 2-D Aliasing Examples

In 2-D, aliasing can change both the **frequency** and the **orientation** of patterns.

In [ ]:
# ── 2-D aliasing: rolling pattern below Nyquist ─────────────────────────────
size = 128; delta_x = 15
x = np.arange(size); y = np.arange(size)
X, Y = np.meshgrid(x, y)

# Diagonal frequencies chosen so x-freq exceeds Nyquist at Δx=15
x_freq, y_freq = 0.05, 0.02
img_original = np.sin(2 * np.pi * (x_freq * X + y_freq * Y))

# Sinc kernel for reconstruction
kx = np.arange(-size//2, size//2); ky = np.arange(-size//2, size//2)
KX, KY = np.meshgrid(kx, ky)
sinc_kernel = np.sinc(KX / delta_x) * np.sinc(KY / delta_x)

# Sample and reconstruct
sampling_mask = np.zeros((size, size))
sampling_mask[::delta_x, ::delta_x] = 1
img_sampled = img_original * sampling_mask
img_recon   = fftconvolve(img_sampled, sinc_kernel, mode='same')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['(a) Original pattern', '(b) Sinc kernel (Δx=15)',
          '(c) Sampled (Δx=15)', '(d) Reconstruction (aliased!)']
images = [img_original, sinc_kernel, img_sampled, img_recon]
for ax, im, t in zip(axes, images, titles):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=10); ax.axis('off')
plt.suptitle('Under-sampling changes both frequency AND orientation', fontsize=12)
plt.tight_layout(); plt.show()

## 1.5 Zone Plate: Variable Frequency Content

In [ ]:
# ── Zone plate: aliasing where local frequency exceeds Nyquist ───────────────
size = 256; delta_x = 5
x = np.arange(-size//2, size//2); y = np.arange(-size//2, size//2)
X, Y = np.meshgrid(x, y)

zone_plate = np.sin(0.005 * (X**2 + Y**2))
sinc_k = np.sinc(X / delta_x) * np.sinc(Y / delta_x)
mask = np.zeros((size, size)); mask[::delta_x, ::delta_x] = 1
sampled = zone_plate * mask
recon   = fftconvolve(sampled, sinc_k, mode='same')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, t in zip(axes,
        [zone_plate, sinc_k, sampled, recon],
        ['(a) Zone plate', '(b) Sinc (Δx=5)', '(c) Sampled', '(d) Reconstructed']):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=10); ax.axis('off')
plt.suptitle('Zone plate: aliasing only where the local frequency exceeds Nyquist',
             fontsize=12)
plt.tight_layout(); plt.show()

## 1.6 Anti-Aliasing: Gaussian Pre-Filter

**The golden rule:** if you can't increase $f_s$, you must decrease $\omega_{\max}$ *before* sampling.

**Standard pipeline:** Gaussian blur ($\sigma \approx \text{factor}/2$) → subsample.

In [ ]:
# ── Anti-aliasing on the zone plate ──────────────────────────────────────────
sigma_blur = 4
zone_blurred = gaussian_filter(zone_plate, sigma=sigma_blur)
sampled_aa = zone_blurred * mask
recon_aa   = fftconvolve(sampled_aa, sinc_k, mode='same')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
titles_top = ['Original zone plate', 'Sampled (no filter)', 'Reconstructed (aliased)']
titles_bot = ['Gaussian-blurred (σ=4)', 'Sampled (after blur)', 'Reconstructed (clean)']
for ax, im, t in zip(axes[0], [zone_plate, sampled, recon], titles_top):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
for ax, im, t in zip(axes[1], [zone_blurred, sampled_aa, recon_aa], titles_bot):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')

plt.suptitle('Anti-aliasing: blur removes high frequencies BEFORE sampling',
             fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Anti-aliasing on a real image ────────────────────────────────────────────
def downsample(img, n):
    """Naive pixel-drop downsampling."""
    return img[::n, ::n]

def downsample_aa(img, n, sigma):
    """Standard pipeline: Gaussian blur then subsample."""
    return gaussian_filter(img, sigma=sigma)[::n, ::n]

factor = 4
sigma_rule = factor / 2   # rule of thumb

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(IMG, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Original {IMG.shape}')

naive = downsample(IMG, factor)
axes[1].imshow(naive, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Naive ↓{factor}× (aliased)')

for i, s in enumerate([sigma_rule, sigma_rule * 2]):
    aa = downsample_aa(IMG, factor, s)
    axes[i+2].imshow(aa, cmap='gray', vmin=0, vmax=1)
    axes[i+2].set_title(f'AA ↓{factor}× σ={s:.1f}')

for ax in axes: ax.axis('off')
plt.suptitle('Rule of thumb: σ ≈ factor / 2', fontsize=13)
plt.tight_layout(); plt.show()

---
### ✏️ Exercise 1 — Checkerboard Aliasing (~10 min)

**Tasks**
1. Create a 256×256 checkerboard (`cell_size=8`).
2. Downsample by 4× **without** and **with** Gaussian pre-filtering (try σ = 1, 2, 4).
3. Display results side by side (spatial domain, top row) and their **2-D Fourier magnitude** (bottom row).
4. **Question:** At which σ does the Moiré pattern disappear? Why?

In [ ]:
# ── Exercise 1: Checkerboard aliasing ────────────────────────────────────────
def make_checkerboard(size=256, cell=8):
    y, x = np.indices((size, size))
    return ((x // cell + y // cell) % 2).astype(np.float32)

checker = make_checkerboard(256, 8)
factor  = 4
sigmas  = [1, 2, 4]
naive   = checker[::factor, ::factor]
aas     = [gaussian_filter(checker, s)[::factor, ::factor] for s in sigmas]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# Top row: spatial domain
all_imgs  = [checker, naive] + aas
all_titles = ['Original', f'Naive ↓{factor}× (Moiré!)'] + \
             [f'σ={s} then ↓{factor}×' for s in sigmas]

for col, (im, title) in enumerate(zip(all_imgs, all_titles)):
    axes[0, col].imshow(im, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=9); axes[0, col].axis('off')

# Bottom row: Fourier magnitude (log scale)
for col, im in enumerate(all_imgs):
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(im))))
    axes[1, col].imshow(mag, cmap='inferno')
    axes[1, col].set_title('|FFT|', fontsize=9); axes[1, col].axis('off')

axes[0, 0].set_ylabel('Spatial'); axes[1, 0].set_ylabel('Fourier')
plt.suptitle('Exercise 1 — Checkerboard: Moiré artefacts vs. anti-aliasing',
             fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# TODO: Answer the question — at which sigma does aliasing vanish?
#       Hint: relate the checkerboard's spatial frequency to the Nyquist limit
#       after downsampling by factor 4.

---
# Section 2 — Quantization

After sampling, each pixel is mapped to a **finite set of levels**: $Q(f) \in \{q_1, \dots, q_L\}$.

With $B$ bits we get $L = 2^B$ levels and a quantization step $\Delta = (f_{\max} - f_{\min}) / L$.

**False contours** (banding) appear when $\Delta$ is large relative to the local gradient.

In [ ]:
# ── Uniform quantization utility ─────────────────────────────────────────────
def quantize(img, bits):
    """Quantize a float [0,1] image to `bits` bit depth."""
    L = 2 ** bits
    return np.clip(np.floor(img * L) / L, 0, 1).astype(np.float32)

def quant_rms(img, bits):
    return float(np.sqrt(np.mean((img - quantize(img, bits))**2)))

# ── Banding demo on a smooth gradient ────────────────────────────────────────
gradient = np.tile(np.linspace(0, 1, 256, dtype=np.float32), (64, 1))
bit_depths = [8, 4, 3, 2]

fig, axes = plt.subplots(len(bit_depths) + 1, 1, figsize=(12, 6), sharex=True)
axes[0].imshow(gradient, cmap='gray', vmin=0, vmax=1, aspect='auto')
axes[0].set_ylabel('float32\n(ref)', fontsize=9); axes[0].set_yticks([])

for ax, bits in zip(axes[1:], bit_depths):
    ax.imshow(quantize(gradient, bits), cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax.set_ylabel(f'{bits}-bit\n({2**bits} lvl)', fontsize=9); ax.set_yticks([])

plt.suptitle('Uniform quantization — banding on a smooth gradient', fontsize=13)
plt.tight_layout(); plt.show()

## 2.1 Image Histograms

The histogram $h(r_k) = n_k$ counts the number of pixels at each intensity level.
The **normalised histogram** $p(r_k) = n_k / (MN)$ is a probability distribution.

Histograms reveal contrast, dynamic range, saturation, and compression artefacts.

In [ ]:
# ── Quantization on the test image + histograms ──────────────────────────────
bit_depths = [8, 4, 2]
images_q = {b: quantize(IMG, b) for b in bit_depths}

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.3)

ax = fig.add_subplot(gs[0, 0])
ax.imshow(IMG, cmap='gray', vmin=0, vmax=1); ax.set_title('Original'); ax.axis('off')

for col, bits in enumerate(bit_depths, 1):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(images_q[bits], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{bits}-bit ({2**bits} lvl)\nRMS={quant_rms(IMG, bits):.4f}')
    ax.axis('off')

# Histograms
ax = fig.add_subplot(gs[1, 0])
ax.hist(IMG.ravel(), bins=256, color='steelblue', alpha=0.7)
ax.set_title('Original histogram'); ax.set_xlim(0, 1)

for col, bits in enumerate(bit_depths, 1):
    ax = fig.add_subplot(gs[1, col])
    ax.hist(images_q[bits].ravel(), bins=256, color='steelblue', alpha=0.7)
    ax.set_title(f'{bits}-bit histogram'); ax.set_xlim(0, 1)

plt.suptitle('Uniform quantization — image quality and histograms', fontsize=13)
plt.tight_layout(); plt.show()

## 2.2 Lloyd–Max Quantization

Uniform bins are optimal **only for uniform distributions**.
Lloyd–Max adapts bin sizes to the image histogram by iterating two rules:

1. **Centroid step:** $q_k = \frac{\int_{z_k}^{z_{k+1}} z\, p(z)\, dz}{\int_{z_k}^{z_{k+1}} p(z)\, dz}$

2. **Midpoint step:** $z_k = \frac{1}{2}(q_{k-1} + q_k)$

In [ ]:
# ── Lloyd–Max quantizer (histogram-driven) ───────────────────────────────────
def lloyd_max_histogram(img, n_levels=8, n_iter=50, n_bins=1024, eps=1e-7):
    """Lloyd-Max quantizer using the image histogram."""
    vals = img.ravel().astype(np.float64)
    hist_counts, bin_edges = np.histogram(vals, bins=n_bins, range=(0, 1))
    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    p = hist_counts.astype(np.float64)

    # Initialise: uniform boundaries
    bounds = np.linspace(0, 1, n_levels + 1)

    for it in range(n_iter):
        # Centroid step
        levels = np.empty(n_levels)
        for k in range(n_levels):
            mask = (bin_centres >= bounds[k]) & (bin_centres < bounds[k + 1])
            mass = p[mask].sum()
            levels[k] = (p[mask] * bin_centres[mask]).sum() / mass if mass > eps else \
                        0.5 * (bounds[k] + bounds[k + 1])
        # Midpoint step
        new_bounds = np.concatenate([[0], 0.5 * (levels[:-1] + levels[1:]), [1]])
        if np.max(np.abs(new_bounds - bounds)) < eps:
            break
        bounds = new_bounds

    # Apply to image
    idx = np.digitize(vals, bounds[1:-1])
    q_img = levels[np.clip(idx, 0, n_levels - 1)].reshape(img.shape)
    return q_img.astype(np.float32), bounds, levels

# Run Lloyd–Max at the same number of levels as 3-bit uniform
n_levels = 8
lm_img, bounds, levels = lloyd_max_histogram(IMG, n_levels=n_levels)
uni_img = quantize(IMG, 3)

rms_uni = float(np.sqrt(np.mean((IMG - uni_img)**2)))
rms_lm  = float(np.sqrt(np.mean((IMG - lm_img)**2)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(IMG,     cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original')
axes[1].imshow(uni_img, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Uniform 3-bit\nRMS = {rms_uni:.4f}')
axes[2].imshow(lm_img,  cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Lloyd–Max (8 lvl)\nRMS = {rms_lm:.4f}')
for ax in axes: ax.axis('off')
plt.suptitle(f'Lloyd–Max reduces RMS by {(1 - rms_lm/rms_uni)*100:.0f}% '
             f'at the same number of levels', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── Decision boundaries on the image histogram ──────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(IMG.ravel(), bins=256, density=True, color='steelblue',
        alpha=0.6, label='Image histogram p(z)')

# Uniform boundaries
uni_bounds = np.linspace(0, 1, n_levels + 1)
for i, b in enumerate(uni_bounds[1:-1]):
    ax.axvline(b, color='tomato', lw=1, ls='--', alpha=0.7,
               label='Uniform boundaries' if i == 0 else None)

# Lloyd–Max boundaries and levels
for i, b in enumerate(bounds[1:-1]):
    ax.axvline(b, color='darkgreen', lw=1.5, ls='-', alpha=0.8,
               label='Lloyd–Max boundaries' if i == 0 else None)
for i, q in enumerate(levels):
    ax.axvline(q, color='gold', lw=1, ls=':',
               label='Lloyd–Max levels' if i == 0 else None)

ax.set_xlabel('Pixel value z'); ax.set_ylabel('p(z)')
ax.set_title('Uniform vs Lloyd–Max: boundary placement on the histogram')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

---
### ✏️ Exercise 2 — Lloyd–Max on a Bimodal Image (~15 min)

A bimodal histogram (e.g. a silhouette or a brain MRI slice) has two intensity peaks.
Uniform quantization wastes levels in the gap between peaks.

**Tasks**
1. Create a synthetic bimodal image (code skeleton below).
2. Quantize it with **uniform 3-bit** and **Lloyd–Max (8 levels)**.
3. Plot both results and their histograms.
4. **Question:** Where do the Lloyd–Max boundaries cluster? Why does this improve RMS?

In [ ]:
# ── Exercise 2: Bimodal image ────────────────────────────────────────────────
# Step 1: Create a bimodal test image
np.random.seed(42)
bimodal = np.zeros((256, 256), dtype=np.float32)
bimodal[:128, :] = np.clip(0.25 + 0.06 * np.random.randn(128, 256), 0, 1)
bimodal[128:, :] = np.clip(0.75 + 0.06 * np.random.randn(128, 256), 0, 1)

# Step 2: Quantize with both methods
uni_bimodal = quantize(bimodal, 3)
lm_bimodal, lm_bounds_b, lm_levels_b = lloyd_max_histogram(bimodal, n_levels=8)

rms_uni_b = float(np.sqrt(np.mean((bimodal - uni_bimodal)**2)))
rms_lm_b  = float(np.sqrt(np.mean((bimodal - lm_bimodal)**2)))

# Step 3: Plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(bimodal,     cmap='gray', vmin=0, vmax=1); axes[0, 0].set_title('Original (bimodal)')
axes[0, 1].imshow(uni_bimodal, cmap='gray', vmin=0, vmax=1); axes[0, 1].set_title(f'Uniform 3-bit\nRMS={rms_uni_b:.4f}')
axes[0, 2].imshow(lm_bimodal,  cmap='gray', vmin=0, vmax=1); axes[0, 2].set_title(f'Lloyd–Max 8 lvl\nRMS={rms_lm_b:.4f}')

for ax in axes[0]: ax.axis('off')

# Histograms with boundaries
for col, (im, title) in enumerate([(bimodal, 'Original'), (uni_bimodal, 'Uniform'), (lm_bimodal, 'Lloyd–Max')]):
    axes[1, col].hist(im.ravel(), bins=256, color='steelblue', alpha=0.6)
    axes[1, col].set_title(title + ' histogram'); axes[1, col].set_xlim(0, 1)

# Show LM boundaries on the original histogram
for b in lm_bounds_b[1:-1]:
    axes[1, 0].axvline(b, color='darkgreen', lw=1.5, ls='-', alpha=0.8)

plt.suptitle('Exercise 2 — Lloyd–Max on a bimodal distribution', fontsize=13)
plt.tight_layout(); plt.show()

# TODO: Answer — where do the Lloyd–Max boundaries cluster, and why?

---
# Section 3 — Interpolation

Interpolation estimates pixel values at **non-integer coordinates** — needed for resizing, rotation, warping, and registration.

## 3.1 1-D Linear Interpolation

Given two samples $Q_0 = f(x_0)$ and $Q_1 = f(x_1)$, the linear estimate at $x$ is:

$$f(x) = (1-t)\,Q_0 + t\,Q_1 \quad\text{where}\quad t = \frac{x - x_0}{x_1 - x_0}$$

In [ ]:
# ── 1-D linear interpolation: step-by-step ───────────────────────────────────
x_nodes = np.array([0.0, 1.0, 3.0, 4.5, 6.0])
y_nodes = np.array([0.3, 0.8, 0.2, 0.9, 0.4])

def linear_interp_1d(x_nodes, y_nodes, x_query):
    """Piecewise linear interpolation from the formula on the slides."""
    out = np.empty_like(x_query)
    for i, xq in enumerate(x_query):
        idx = np.clip(np.searchsorted(x_nodes, xq) - 1, 0, len(x_nodes) - 2)
        x0, x1 = x_nodes[idx], x_nodes[idx + 1]
        Q0, Q1 = y_nodes[idx], y_nodes[idx + 1]
        t = (xq - x0) / (x1 - x0)
        out[i] = (1 - t) * Q0 + t * Q1
    return out

xq = np.linspace(0, 6, 500)
yq = linear_interp_1d(x_nodes, y_nodes, xq)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(xq, yq, 'steelblue', lw=2, label='Linear interpolation')
ax.plot(x_nodes, y_nodes, 'ko', ms=8, zorder=5, label='Known samples')
for xi, yi in zip(x_nodes, y_nodes):
    ax.annotate(f'({xi},{yi})', (xi, yi), textcoords='offset points',
                xytext=(5, 8), fontsize=8)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('1-D Piecewise Linear Interpolation'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3.2 Nearest-Neighbor Interpolation (2-D)

$$I'(x,y) = I\!\left(\lfloor x + 0.5 \rfloor,\; \lfloor y + 0.5 \rfloor\right)$$

Fast ($O(1)$), preserves exact values, but **blocky**. Correct for label maps.

In [ ]:
# ── Nearest-Neighbor from scratch ────────────────────────────────────────────
def nearest_upsample(img, scale):
    H, W = img.shape
    H_out, W_out = int(H * scale), int(W * scale)
    r_src = (np.arange(H_out) + 0.5) / scale - 0.5
    c_src = (np.arange(W_out) + 0.5) / scale - 0.5
    r_nn = np.clip(np.floor(r_src + 0.5).astype(int), 0, H - 1)
    c_nn = np.clip(np.floor(c_src + 0.5).astype(int), 0, W - 1)
    return img[r_nn[:, None], c_nn[None, :]]

small = IMG[::8, ::8]   # 32×32
nn_up = nearest_upsample(small, 8)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(IMG, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original 256×256')
axes[1].imshow(small, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Downsampled 32×32')
axes[2].imshow(nn_up, cmap='gray', vmin=0, vmax=1); axes[2].set_title('NN upsampled 256×256')
for ax in axes: ax.axis('off')
plt.suptitle('Nearest-Neighbor: blocky staircase artefacts', fontsize=13)
plt.tight_layout(); plt.show()

## 3.3 Bilinear Interpolation (2-D)

Three linear interpolations: 2 horizontal + 1 vertical.

$$f(x,y) = (1-\alpha)(1-\beta)\,Q_{11} + \alpha(1-\beta)\,Q_{21} + (1-\alpha)\beta\,Q_{12} + \alpha\beta\,Q_{22}$$

In [ ]:
# ── Bilinear: step-by-step visualisation on a 2×2 cell ──────────────────────
import matplotlib.patches as mpatches

Q00, Q10, Q01, Q11 = 0.1, 0.8, 0.4, 0.6
x0, x1, y0, y1 = 0.0, 1.0, 0.0, 1.0
xq, yq = 0.4, 0.7

alpha = (xq - x0) / (x1 - x0)
beta  = (yq - y0) / (y1 - y0)
R0 = (1 - alpha) * Q00 + alpha * Q10       # along y0
R1 = (1 - alpha) * Q01 + alpha * Q11       # along y1
Iq = (1 - beta) * R0 + beta * R1           # vertical

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-0.3, 1.5); ax.set_ylim(-0.3, 1.5); ax.set_aspect('equal')
# Grid
ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], 'k-', lw=1.5)
# Corner values
for (cx, cy, val) in [(0,0,Q00),(1,0,Q10),(0,1,Q01),(1,1,Q11)]:
    ax.plot(cx, cy, 'ko', ms=10)
    ax.annotate(f'Q={val}', (cx, cy), textcoords='offset points',
                xytext=(8, 8), fontsize=11, fontweight='bold')
# Intermediate points
ax.plot(xq, 0, 's', color='teal', ms=10)
ax.annotate(f'R0={R0:.2f}', (xq, 0), textcoords='offset points',
            xytext=(8, -15), fontsize=10, color='teal')
ax.plot(xq, 1, 's', color='purple', ms=10)
ax.annotate(f'R1={R1:.2f}', (xq, 1), textcoords='offset points',
            xytext=(8, 8), fontsize=10, color='purple')
# Query point
ax.plot(xq, yq, '*', color='red', ms=15, zorder=5)
ax.annotate(f'f({xq},{yq})={Iq:.3f}', (xq, yq), textcoords='offset points',
            xytext=(12, 0), fontsize=11, color='red', fontweight='bold')
# Lines
ax.plot([xq, xq], [0, 1], '--', color='steelblue', lw=1.5)
ax.set_title(f'Bilinear interpolation step-by-step\n'
             f'α={alpha:.1f}, β={beta:.1f}', fontsize=12)
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

In [ ]:
# ── Bilinear from scratch — full image upsample ─────────────────────────────
def bilinear_upsample(img, scale):
    """Bilinear upsampling (pure NumPy, matches the slide derivation)."""
    H, W = img.shape
    H_out, W_out = int(H * scale), int(W * scale)
    r_src = np.clip((np.arange(H_out) + 0.5) / scale - 0.5, 0, H - 1.0001)
    c_src = np.clip((np.arange(W_out) + 0.5) / scale - 0.5, 0, W - 1.0001)
    r0 = np.floor(r_src).astype(int); dr = (r_src - r0)[:, None]
    c0 = np.floor(c_src).astype(int); dc = (c_src - c0)[None, :]
    r1 = np.minimum(r0 + 1, H - 1); c1 = np.minimum(c0 + 1, W - 1)
    return ((1-dr)*(1-dc) * img[r0[:,None], c0[None,:]] +
            (1-dr)*dc     * img[r0[:,None], c1[None,:]] +
            dr*(1-dc)     * img[r1[:,None], c0[None,:]] +
            dr*dc         * img[r1[:,None], c1[None,:]]).astype(np.float32)

bl_up = bilinear_upsample(small, 8)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(small, cmap='gray', vmin=0, vmax=1); axes[0].set_title('32×32 input')
axes[1].imshow(nn_up, cmap='gray', vmin=0, vmax=1); axes[1].set_title('NN 256×256')
axes[2].imshow(bl_up, cmap='gray', vmin=0, vmax=1); axes[2].set_title('Bilinear 256×256')
for ax in axes: ax.axis('off')
plt.suptitle('NN vs Bilinear: bilinear removes the blocky artefacts', fontsize=13)
plt.tight_layout(); plt.show()

## 3.4 Bicubic & Spline Interpolation

**Bicubic** uses a 4×4 neighbourhood and a cubic weight function → sharper, but may ring.

**Spline** ($C^2$ smooth) is the gold standard for medical image registration.

In [ ]:
# ── Full comparison: NN / Bilinear / Bicubic / Spline ────────────────────────
def resize_method(img, out_shape, method):
    zy = out_shape[0] / img.shape[0]; zx = out_shape[1] / img.shape[1]
    orders = {'nearest': 0, 'bilinear_scratch': 1, 'bicubic': 3, 'spline': 5}
    if method == 'bilinear_scratch':
        return bilinear_upsample(img, zy)
    return scipy_zoom(img, (zy, zx), order=orders[method]).astype(np.float32)

methods = ['nearest', 'bilinear_scratch', 'bicubic', 'spline']
labels  = ['Nearest Neighbor', 'Bilinear (scratch)', 'Bicubic', 'Spline (B5)']
results = {}; times_ms = {}

for m in methods:
    t0 = time.perf_counter()
    results[m] = resize_method(small, IMG.shape, m)
    times_ms[m] = (time.perf_counter() - t0) * 1000

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(IMG, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Ground truth')
for ax, m, lbl in zip(axes[1:], methods, labels):
    rmse = float(np.sqrt(np.mean((IMG - results[m])**2)))
    ax.imshow(results[m], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{lbl}\nRMSE={rmse:.4f}  {times_ms[m]:.0f}ms', fontsize=9)
for ax in axes: ax.axis('off')
plt.suptitle('Interpolation comparison: 32×32 → 256×256', fontsize=13)
plt.tight_layout(); plt.show()

---
# Section 4 — Image Pyramids & Multi-scale Analysis

## 4.1 Gaussian Pyramid

$$G_0 = \text{original}, \qquad G_{k+1} = \mathrm{DOWN}(G_k * h_\sigma)$$

Each level: blur → subsample ↓2. Level $k$ has $1/4$ the pixels of level $k-1$.

## 4.2 Laplacian Pyramid

$$L_k = G_k - \mathrm{UP}(G_{k+1}), \qquad L_N = G_N$$

Stores **band-pass residuals** (details lost during blurring). Allows **perfect reconstruction**.

In [ ]:
# ── Gaussian & Laplacian pyramids ────────────────────────────────────────────
def build_gaussian_pyramid(img, n=5, sigma=1.2):
    pyr = [img.copy()]
    for _ in range(n - 1):
        pyr.append(gaussian_filter(pyr[-1], sigma)[::2, ::2])
    return pyr

def upsample2x(img, target_shape):
    return scipy_zoom(img, (target_shape[0]/img.shape[0],
                            target_shape[1]/img.shape[1]),
                      order=1).astype(np.float32)

def build_laplacian_pyramid(gpyr):
    lpyr = [gpyr[k] - upsample2x(gpyr[k+1], gpyr[k].shape)
            for k in range(len(gpyr) - 1)]
    lpyr.append(gpyr[-1])
    return lpyr

def reconstruct(lpyr):
    img = lpyr[-1].copy()
    for k in range(len(lpyr) - 2, -1, -1):
        img = lpyr[k] + upsample2x(img, lpyr[k].shape)
    return img

# Build pyramids
gpyr = build_gaussian_pyramid(IMG, n=5)
lpyr = build_laplacian_pyramid(gpyr)

# Display Gaussian pyramid
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for i, g in enumerate(gpyr):
    axes[i].imshow(g, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'G{i}  {g.shape[0]}×{g.shape[1]}', fontsize=10)
    axes[i].axis('off')
plt.suptitle('Gaussian Pyramid — progressive blur + downsample', fontsize=13)
plt.tight_layout(); plt.show()

# Display Laplacian pyramid
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for i, l in enumerate(lpyr):
    if i < len(lpyr) - 1:
        vm = np.percentile(np.abs(l), 99)
        axes[i].imshow(l, cmap='bwr', vmin=-vm, vmax=vm)
        axes[i].set_title(f'L{i} (band-pass)  {l.shape[0]}×{l.shape[1]}', fontsize=10)
    else:
        axes[i].imshow(l, cmap='gray', vmin=0, vmax=1)
        axes[i].set_title(f'L{i} (low-pass base)  {l.shape[0]}×{l.shape[1]}', fontsize=10)
    axes[i].axis('off')
plt.suptitle('Laplacian Pyramid — band-pass residuals (red = +, blue = −)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── Perfect reconstruction verification ──────────────────────────────────────
recon = np.clip(reconstruct(lpyr), 0, 1)
rmse = float(np.sqrt(np.mean((IMG - recon)**2)))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(IMG,   cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original')
axes[1].imshow(recon, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Reconstructed from Laplacian\nRMSE = {rmse:.2e}')
axes[2].imshow(np.abs(IMG - recon) * 500, cmap='hot', vmin=0, vmax=1)
axes[2].set_title('|Difference| × 500')
for ax in axes: ax.axis('off')
plt.suptitle('Perfect Reconstruction — errors at machine precision', fontsize=13)
plt.tight_layout(); plt.show()

---
## Lecture Summary

| Topic | Key Take-away |
|---|---|
| Image formation | $I = E \cdot R$; sensor integrates via PSF convolution |
| Fourier / Sampling | Sampling replicates the spectrum; aliasing = spectral overlap |
| Nyquist condition | $f_s \geq 2 f_{\max}$ ↔ $\Delta x \leq 1/(2\omega_{\max})$ |
| Anti-aliasing | Gaussian low-pass before downsampling ($\sigma \approx \text{factor}/2$) |
| Uniform quantization | $\Delta / \sqrt{12}$ RMS error; banding below ~4 bits |
| Lloyd–Max | Adapts bins to histogram → lower MSE for the same $L$ |
| Interpolation | NN (fast, blocky) → Bilinear (smooth) → Bicubic (sharp) → Spline ($C^2$) |
| Image pyramids | Gaussian (blur+↓2) and Laplacian (band-pass residual, perfect recon) |

---
## References

1. **Gonzalez, R.C. & Woods, R.E.** (2018). *Digital Image Processing* (4th ed.). Pearson.
2. **Burger, W. & Burge, M.** (2016). *Digital Image Processing: An Algorithmic Introduction Using Java* (2nd ed.). Springer.
3. **Burt, P.J. & Adelson, E.H.** (1983). *The Laplacian pyramid as a compact image code.* IEEE Trans. Commun., 3(4).
4. **Lloyd, S.P.** (1982). *Least squares quantization in PCM.* IEEE Trans. Inf. Theory, 28(2), 129–137.